# Exploratory Analysis — Supply Chain Shortage Evaluation Framework

This notebook documents the first stage of the project: understanding the M5 Forecasting dataset and deciding how to represent the data for a shortage-risk evaluation framework.

The key methodological point is that the dataset does not contain direct inventory or stockout labels. Because of that, we begin by studying sales behavior and defining a proxy target for **inventory stress** based on unusually high demand.


## 1. Environment and package check

Before touching the data, we verify that the core Python packages load correctly. This confirms that the local VS Code/Jupyter environment is ready for analysis.

At this stage, we are not modeling yet. We are only checking that the notebook runtime can import the libraries needed for data analysis and later baseline modeling.


In [9]:
import pandas as pd
import numpy as np
import sklearn
import xgboost

print("All packages loaded")

All packages loaded


## 2. Load the raw M5 dataset files

The dataset is stored in `data/raw/`. We load three core files:

- `calendar.csv`: maps day numbers like `d_1`, `d_2`, etc. to actual dates, events, and SNAP indicators.
- `sell_prices.csv`: contains item prices by store and week.
- `sales_train_validation.csv`: contains daily unit sales for each item-store combination.

The `.shape` output gives a first sense of scale: number of rows and columns in each file.


In [10]:
from pathlib import Path

RAW_DIR = Path("../data/raw")

calendar = pd.read_csv(RAW_DIR / "calendar.csv")
prices = pd.read_csv(RAW_DIR / "sell_prices.csv")
sales = pd.read_csv(RAW_DIR / "sales_train_validation.csv")

print("calendar:", calendar.shape)
print("prices:", prices.shape)
print("sales:", sales.shape)

calendar: (1969, 14)
prices: (6841121, 4)
sales: (30490, 1919)


## 3. Inspect column structure

We inspect column names to understand how the dataset is organized.

The sales file is in **wide format**: one row per item-store combination, with each day stored as a separate column (`d_1` through `d_1913`). This is convenient for compact storage, but less convenient for feature engineering and machine learning pipelines.


In [11]:
print("Calendar columns:")
print(calendar.columns.tolist())

print("\nPrices columns:")
print(prices.columns.tolist())

print("\nSales columns:")
print(sales.columns.tolist()[:20])
print("...")
print(sales.columns.tolist()[-10:])

Calendar columns:
['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

Prices columns:
['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

Sales columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9', 'd_10', 'd_11', 'd_12', 'd_13', 'd_14']
...
['d_1904', 'd_1905', 'd_1906', 'd_1907', 'd_1908', 'd_1909', 'd_1910', 'd_1911', 'd_1912', 'd_1913']


## 4. Understand dataset hierarchy

We count the number of unique items, stores, categories, and departments.

This tells us the product hierarchy and retail coverage of the dataset. It also helps us decide whether to model at the item level, store level, category level, or some sampled subset during early experimentation.


In [12]:
print("Unique items:", sales["item_id"].nunique())
print("Unique stores:", sales["store_id"].nunique())
print("Unique categories:", sales["cat_id"].nunique())
print("Unique departments:", sales["dept_id"].nunique())

Unique items: 3049
Unique stores: 10
Unique categories: 3
Unique departments: 7


## 5. Preview the sales table

We preview the first few rows to see the actual layout of the sales data.

Each row represents a single item-store series. The daily sales values are spread across many columns, which is why the next step reshapes the table.


In [18]:
sales.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


## 6. Reshape sales from wide format to long format

We use `pandas.melt()` to convert the table from wide format to long format.

Before melting:

- One row = one item-store combination
- Each day is a separate column

After melting:

- One row = one item-store-day observation
- The day is stored in a `day` column
- The daily unit sales value is stored in a `sales` column

This long format is much easier for building features such as rolling averages, lagged sales, demand spikes, and future stress-event labels.


In [19]:
sales_long = sales.melt(
    id_vars=[
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ],
    var_name="day",
    value_name="sales"
)

sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


## 7. Measure full long-format size and memory cost

After melting, we check the resulting shape and memory usage.

This is an important data engineering step. The full long-format dataset becomes very large, so we need to decide whether to continue with all rows or create a smaller development subset.


In [21]:
sales_long.shape
sales_long.memory_usage(deep=True).sum() / 1024**3

np.float64(22.782371260225773)

## 8. Create a manageable development subset

The full melted dataset used about 22.8 GB of memory, which is too large for comfortable experimentation.

To develop the evaluation methodology efficiently, we select the first 100 unique items and keep their store-level sales rows. This gives us a representative but much smaller working dataset.

This is not the final research claim. It is a practical development subset used to design and test the pipeline before scaling.


In [22]:
sample_items = sales["item_id"].unique()[:100]

sales_small = sales[
    sales["item_id"].isin(sample_items)
]

## 9. Check the sampled wide-format dataset size

We verify the size of the sampled dataset before reshaping it.

The expected structure is roughly 100 items across 10 stores, with the same daily sales columns as the full dataset.


In [23]:
sales_small.shape

(1000, 1919)

## 10. Melt the sampled dataset

We reshape only the sampled dataset into long format.

This gives us a much smaller item-store-day table that is practical for exploration, target definition, and early baseline experiments.


In [24]:
sales_small_long = sales_small.melt(
    id_vars=[
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ],
    var_name="day",
    value_name="sales"
)

## 11. Check sampled long-format row count

We confirm the number of rows in the sampled long-format dataset.

This tells us how many item-store-day observations we can use for early feature engineering and target creation.


In [25]:
sales_small_long.shape

(1913000, 8)

## 12. Check sampled memory usage

We measure the memory footprint of the sampled long-format dataset.

This helps confirm that the subset is suitable for interactive notebook work in VS Code.


In [26]:
sales_small_long.memory_usage(deep=True).sum() / 1024**3

np.float64(0.7490312047302723)

## 13. Delete the full melted dataframe

Since the full `sales_long` dataframe consumed too much memory, we delete it after measuring its size.

This keeps the notebook responsive and prevents later cells from running out of memory.


In [27]:
del sales_long

## 14. Trigger garbage collection

After deleting the large dataframe, we manually run Python garbage collection.

This asks Python to reclaim unused memory so the notebook remains stable during further analysis.


In [28]:
import gc
gc.collect()

89

## 15. Reconfirm sampled wide-format size

We re-check `sales_small.shape` after freeing memory to confirm that our smaller working dataset is still available.


In [30]:
sales_small.shape

(1000, 1919)

## 16. Reconfirm sampled long-format size

We re-check `sales_small_long.shape` to verify that the smaller long-format table is still available for analysis.


In [31]:
sales_small_long.shape


(1913000, 8)

## 17. Reconfirm sampled memory usage

We measure memory again to ensure the working dataset remains manageable.

This reinforces the key project decision: develop the methodology on a smaller subset first, then scale later if needed.


In [32]:
sales_small_long.memory_usage(deep=True).sum() / 1024**3

np.float64(0.7490312047302723)

## 18. Preview sampled long-format data

We inspect the first few rows of the sampled long-format dataset.

At this point, each row should represent one item-store-day observation with columns for product identifiers, store identifiers, day, and sales.


In [33]:
sales_small_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


## 19. Explore sales distribution by item

We summarize daily sales distribution for each item.

This helps us understand what “normal” and “unusually high” sales look like for each product. This matters because different products have very different typical sales volumes.


In [34]:
sales_small_long.groupby("item_id")["sales"].describe()

,count,mean,std,min,25%,50%,75%,max
item_id,,,,,,,,
HOBBIES_1_001,19130.0,0.213957,0.576033,0.0,0.0,0.0,0.0,6.0
HOBBIES_1_002,19130.0,0.264454,0.593988,0.0,0.0,0.0,0.0,11.0
HOBBIES_1_003,19130.0,0.075013,0.323133,0.0,0.0,0.0,0.0,6.0
HOBBIES_1_004,19130.0,2.047831,2.668301,0.0,0.0,1.0,3.0,25.0
HOBBIES_1_005,19130.0,0.764297,1.225230,0.0,0.0,0.0,1.0,15.0
...,...,...,...,...,...,...,...,...
HOBBIES_1_099,19130.0,0.791375,1.207007,0.0,0.0,0.0,1.0,9.0
HOBBIES_1_100,19130.0,0.396864,0.694519,0.0,0.0,0.0,1.0,7.0
HOBBIES_1_102,19130.0,0.101045,0.360911,0.0,0.0,0.0,0.0,8.0


## 20. Define initial demand-stress thresholds

Because the M5 dataset does not include inventory levels or true stockout labels, we define a proxy target.

Initial proxy definition:

> A demand-stress event occurs when daily sales exceed the item's historical 90th percentile.

This threshold identifies unusually high demand days for each item. It is not a perfect shortage label, but it is a transparent and reproducible proxy for inventory pressure.

The next step will be to convert these thresholds into a binary `stress_event` label.


In [35]:
stress_thresholds = (
    sales_small_long
    .groupby("item_id")["sales"]
    .quantile(0.90)
)